In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from statsmodels.stats.inter_rater import fleiss_kappa
import warnings
warnings.filterwarnings('ignore')



In [2]:
# 1) Load
df = pd.read_csv('TAZEEN RASHID - Material_Takehome_email_phishing_data_Labeling_ORIGINAL - email_phishing_data.csv', dtype=str)

In [3]:
df

,num_words,num_unique_words,num_stopwords,num_links,num_unique_domains,num_email_addresses,num_spelling_errors,num_urgent_keywords,label_1,label_2,label_3
0,119,72,30,0,0,0,11,0,0,NaN,NaN
1,34,27,7,1,1,0,2,0,0,NaN,NaN
2,381,194,134,0,0,0,28,1,0,NaN,NaN
3,8,8,3,0,0,0,1,0,0,NaN,NaN
4,2671,1055,466,48,9,2,261,2,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
524841,224,143,93,0,0,0,5,0,0,NaN,NaN
524842,51,34,4,0,0,0,7,0,0,NaN,NaN
524843,85,35,27,0,0,0,11,0,0,NaN,NaN
524844,8,8,3,0,0,0,0,0,0,NaN,NaN


In [4]:
# Find all FALSE values in the DataFrame
false_values = df == "FALSE"

# Total number of FALSE values
total_false = false_values.sum().sum()
print("Total FALSE values:", total_false)

Total FALSE values: 0


In [5]:
# ensure labels are numeric (0,1 or NaN)
for lab in ['label_1','label_2','label_3']:
    if lab in df.columns:
        df[lab] = pd.to_numeric(df[lab].replace('FALSE', np.nan), errors='coerce')

# Quick datasummary
print("Shape:", df.shape)
print("Labels present counts:\n", df[['label_1','label_2','label_3']].notnull().sum())

Shape: (524846, 11)
Labels present counts:
 label_1    524846
label_2     26949
label_3     12573
dtype: int64


In [6]:
# 2) Pairwise Cohen's kappa (only rows where both judges labeled)
pairs = [('label_1','label_2'),('label_1','label_3'),('label_2','label_3')]
for a,b in pairs:
    both = df[df[a].notnull() & df[b].notnull()]
    if len(both)>0:
        k = cohen_kappa_score(both[a], both[b])
        pct = (both[a]==both[b]).mean()
        print(f"{a} vs {b}: n={len(both)}, Cohen's kappa={k:.3f}, percent_agree={pct:.3%}")
    else:
        print(f"{a} vs {b}: no overlapping labels")

label_1 vs label_2: n=26949, Cohen's kappa=0.759, percent_agree=90.853%
label_1 vs label_3: n=12573, Cohen's kappa=0.769, percent_agree=90.511%
label_2 vs label_3: n=12573, Cohen's kappa=0.749, percent_agree=89.883%


In [9]:


# 1. Select your specific columns
target_cols = ['label_1', 'label_2', 'label_3']

# 2. Convert to numeric all at once (Vectorized)
# .apply() here handles the columns internally without you writing a loop.
# errors='coerce' turns any remaining non-numbers into NaN
df_clean = df[target_cols].apply(pd.to_numeric, errors='coerce')

# 3. Count Votes (Vectorized)
# "axis=1" tells Python to sum across the row, not down the column.
zeros = (df_clean == 0).sum(axis=1)
ones  = (df_clean == 1).sum(axis=1)

# Calculate total judges per row (0s + 1s)
n_judges = zeros + ones

# 4. Create the Matrix [Count_of_0s, Count_of_1s]
mat = np.column_stack((zeros, ones))

# 5. Split and Calculate (No loops, just boolean filtering)
# Filter for rows where exactly 3 judges voted
mat_3 = mat[n_judges == 3]

# Filter for rows where exactly 2 judges voted
mat_2 = mat[n_judges == 2]

print(f"--- Fast Report ---")
if len(mat_3) > 0:
    print(f"Kappa (3 Judges): {fleiss_kappa(mat_3):.4f}")
else:
    print("Kappa (3 Judges): No valid rows")

if len(mat_2) > 0:
    print(f"Kappa (2 Judges): {fleiss_kappa(mat_2):.4f}")
else:
    print("Kappa (2 Judges): No valid rows")

--- Fast Report ---
Kappa (3 Judges): 0.6806
Kappa (2 Judges): 1.0000


In [10]:

from sklearn.metrics import confusion_matrix

def kappa_diagnostics(df, col_a, col_b):
    # Filter for rows where both judges provided a label
    subset = df[[col_a, col_b]].dropna()
    
    # Create Confusion Matrix
    # a = Both Phishing (1,1)
    # b = Judge A Phishing, Judge B Safe (1,0)
    # c = Judge A Safe, Judge B Phishing (0,1)
    # d = Both Safe (0,0)
    # Note: sklearn returns [[TN, FP], [FN, TP]] -> [[d, c], [b, a]]
    cm = confusion_matrix(subset[col_a], subset[col_b], labels=[0, 1])
    d, c, b, a = cm.ravel() 
    
    n = len(subset)
    
    # 1. Prevalence Index (PI)
    # Measures how skewed the data is towards one class.
    # Range: 0 (Balanced) to 1 (All one class).
    # Formula: |Probability(Yes) - Probability(No)|
    pi = abs((a/n) - (d/n))
    
    # 2. Bias Index (BI)
    # Measures if one judge is "trigger happy" compared to the other.
    # Range: 0 (No Bias) to 1 (Total Disagreement on margins).
    # Formula: |Prop(Judge A Yes) - Prop(Judge B Yes)|
    bi = abs((b/n) - (c/n))
    
    print(f"--- Diagnostics for {col_a} vs {col_b} ---")
    print(f"N: {n}")
    print(f"Prevalence Index (PI): {pi:.4f} (High = Imbalanced Data)")
    print(f"Bias Index (BI):       {bi:.4f} (High = Judges Disagree on definitions)")
    print("-" * 30)

# Run it for your primary pair
kappa_diagnostics(df, 'label_1', 'label_2')

--- Diagnostics for label_1 vs label_2 ---
N: 26949
Prevalence Index (PI): 0.4923 (High = Imbalanced Data)
Bias Index (BI):       0.0081 (High = Judges Disagree on definitions)
------------------------------


In [12]:


def analyze_single_judge_risk(df):
    """
    Calculates the False Negative Rate (Miss Rate) of single-judge 'Safe' labels
    using the multi-judge control sample.
    """
    
    # ---------------------------------------------------------
    # STEP 1: Establish the "Ground Truth" (Final Label)
    # ---------------------------------------------------------
    # We need the consensus view to know if Judge 1 was actually wrong.
    label_cols = ['label_1', 'label_2', 'label_3']
    
    # Ensure columns are numeric (handling "FALSE" or strings)
    for col in label_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Calculate Majority Vote
    vote_sum = df[label_cols].sum(axis=1)
    vote_count = df[label_cols].notna().sum(axis=1)
    
    # Final Label: 1 if > 50% of judges said Phishing, else 0
    # Note: We handle division by zero for empty rows just in case
    df['Final_Label'] = np.where(vote_count > 0, (vote_sum / vote_count > 0.5).astype(int), np.nan)

    # ---------------------------------------------------------
    # STEP 2: Isolate the "Audit Set" (The 20k Control Group)
    # ---------------------------------------------------------
    # Definition: Rows where Judge 1 said "Safe" (0) AND a 2nd judge reviewed it.
    # We use .notna() on label_2 to find cases that had a second opinion.
    audit_mask = (df['label_1'] == 0) & (df['label_2'].notna())
    audit_set = df[audit_mask]
    
    audit_size = len(audit_set)

    # ---------------------------------------------------------
    # STEP 3: Identify Misses (False Negatives)
    # ---------------------------------------------------------
    # A "Miss" is when Judge 1 said Safe (0), but the Final Consensus is Phishing (1).
    misses = audit_set[audit_set['Final_Label'] == 1]
    miss_count = len(misses)

    # Calculate the Error Rate
    if audit_size > 0:
        fn_rate = miss_count / audit_size
    else:
        fn_rate = 0.0

    # ---------------------------------------------------------
    # STEP 4: Project Impact on the Unverified Pile
    # ---------------------------------------------------------
    # The "Unverified" pile are rows where Judge 1 said Safe, but NO ONE else checked.
    unverified_mask = (df['label_1'] == 0) & (df['label_2'].isna())
    unverified_count = len(df[unverified_mask])
    
    projected_hidden_phishing = int(unverified_count * fn_rate)

    # ---------------------------------------------------------
    # STEP 5: Generate Risk Report
    # ---------------------------------------------------------
    print(f"=======================================================")
    print(f"   RISK ASSESSMENT: QUANTIFYING VERIFICATION BIAS")
    print(f"=======================================================")
    print(f"1. AUDIT SAMPLE (Double-Checked 'Safe' Emails)")
    print(f"   - Sample Size:           {audit_size:,} emails")
    print(f"   - Judge 1 'Safe' Votes:  {audit_size:,}")
    print(f"   - Consensus Overrules:   {miss_count:,} (True Phishing)")
    print(f"   ---------------------------------------------------")
    print(f"   -> FALSE NEGATIVE RATE:  {fn_rate:.4%} (The 'Miss Rate')")
    print(f"\n2. PROJECTION (Unverified Data)")
    print(f"   - Unverified 'Safe' Pile: {unverified_count:,} emails")
    print(f"   ---------------------------------------------------")
    print(f"   -> ESTIMATED HIDDEN PHISHING: ~{projected_hidden_phishing:,} emails")
    print(f"=======================================================")
    
    return fn_rate, projected_hidden_phishing

# --- EXECUTE THE FUNCTION ---
# Assuming your dataframe is named 'df'
fn_rate, hidden_count = analyze_single_judge_risk(df)

   RISK ASSESSMENT: QUANTIFYING VERIFICATION BIAS
1. AUDIT SAMPLE (Double-Checked 'Safe' Emails)
   - Sample Size:           20,000 emails
   - Judge 1 'Safe' Votes:  20,000
   - Consensus Overrules:   434 (True Phishing)
   ---------------------------------------------------
   -> FALSE NEGATIVE RATE:  2.1700% (The 'Miss Rate')

2. PROJECTION (Unverified Data)
   - Unverified 'Safe' Pile: 497,897 emails
   ---------------------------------------------------
   -> ESTIMATED HIDDEN PHISHING: ~10,804 emails


In [16]:
# 1. Setup Data & Clean Types
cols = ['label_1', 'label_2', 'label_3']
# Ensure numeric (coerces "FALSE" or strings to NaN)
for c in cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# 2. Pre-Calculate Vectors (Speed Optimization)
# Count how many judges actually voted (non-NaN)
n_votes = df[cols].notna().sum(axis=1)
# Sum the votes (treating 1 as Phish, 0 as Safe)
vote_sum = df[cols].sum(axis=1)
# Calculate the ratio of Phish votes
# (We replace 0 with 1 temporarily to avoid DivisionByZero errors, masked later)
phish_ratio = vote_sum / n_votes.replace(0, 1)

# 3. Initialize Columns (Default State)
df['final_label'] = np.nan
df['label_confidence'] = 0.0
df['label_type'] = 'no_judges'  # Default for n_votes == 0

# ---------------------------------------------------------
# RULE IMPLEMENTATION
# ---------------------------------------------------------

# === Rule: "For items with only one judge... set label_confidence = 0.5" ===
mask_single = (n_votes == 1)
# Set label to whatever the single judge said
df.loc[mask_single, 'final_label'] = df.loc[mask_single, cols].max(axis=1)
# Cap confidence at 0.5 (Low Confidence)
df.loc[mask_single, 'label_confidence'] = 0.5
df.loc[mask_single, 'label_type'] = 'single_judge'

# === Rule: "If exactly tie (e.g., 1 phish / 1 safe)... flag as tie_needs_adjudication" ===
# Tie is defined as ratio == 0.5 (only possible with 2 judges or 4 judges)
mask_tie = (n_votes > 1) & (phish_ratio == 0.5)
df.loc[mask_tie, 'label_type'] = 'tie_needs_adjudication'
# final_label remains NaN (as requested)

# === Rule: "If majority exists... set final_label and label_confidence" ===
# Decisive means we have votes, and it's not a tie
mask_decisive = (n_votes > 1) & (phish_ratio != 0.5)

# Calculate Majority Fraction (Confidence)
# If ratio is 0.66 (Phish), conf is 0.66. If ratio is 0.0 (Safe), conf is 1.0.
df.loc[mask_decisive, 'label_confidence'] = np.maximum(phish_ratio, 1 - phish_ratio)

# Set Final Label (1 if >0.5, else 0)
df.loc[mask_decisive, 'final_label'] = (phish_ratio > 0.5).astype(int)

# === Rule: "If tie but label_3 exists... set label_type='adjudicated_by_label3'" ===
# In binary classification (0 vs 1), you cannot have a tie with 3 judges.
# So "adjudicated" means 3 judges voted, but they didn't agree 100% (Split Decision).
mask_adj = mask_decisive & (n_votes == 3) & (phish_ratio != 0.0) & (phish_ratio != 1.0)
df.loc[mask_adj, 'label_type'] = 'adjudicated_by_label3'

# === Rule: "label_type='consensus_*' (all/majority)" ===
# Consensus Phish: Decisive, Majority is 1, NOT adjudicated (implied unanimous or majority 2-judge)
# Note: If you want 'adjudicated' to take precedence for metrics, keep it above.
# If you want 'consensus' to cover split decisions too, change logic.
# Here I strictly separate "Adjudicated" (3 judges split) from "Consensus" (Unanimous or 2-judge Majority).

# Consensus Phish (Majority > 0.5 and NOT the adjudicated 3-judge cases)
mask_phish = mask_decisive & (~mask_adj) & (phish_ratio > 0.5)
df.loc[mask_phish, 'label_type'] = 'consensus_phish'

# Consensus Safe (Majority < 0.5 and NOT the adjudicated 3-judge cases)
mask_safe = mask_decisive & (~mask_adj) & (phish_ratio < 0.5)
df.loc[mask_safe, 'label_type'] = 'consensus_safe'

# ---------------------------------------------------------
# VERIFICATION
# ---------------------------------------------------------
print("--- Label Type Distribution ---")
print(df['label_type'].value_counts())

print("\n--- Confidence Check (First 5 Single Judges) ---")
print(df[df['label_type'] == 'single_judge'][['final_label', 'label_confidence']].head())

print("\n--- Adjudicated Check (First 5 Hard Cases) ---")
print(df[df['label_type'] == 'adjudicated_by_label3'][['final_label', 'label_confidence']].head())

--- Label Type Distribution ---
label_type
single_judge             497897
consensus_safe            18876
consensus_phish            5608
adjudicated_by_label3      2465
Name: count, dtype: int64

--- Confidence Check (First 5 Single Judges) ---
   final_label  label_confidence
0          0.0               0.5
1          0.0               0.5
2          0.0               0.5
3          0.0               0.5
5          0.0               0.5

--- Adjudicated Check (First 5 Hard Cases) ---
     final_label  label_confidence
350          1.0          0.666667
449          1.0          0.666667
491          0.0          0.666667
526          1.0          0.666667
858          0.0          0.666667


In [18]:
# 1. Define the specific new columns we just created
new_ml_cols = ['final_label', 'label_confidence', 'label_type']

# 2. Get the rest of the columns (whatever they currently are)
# This filters out the new cols so we don't duplicate them
remaining_cols = [c for c in df.columns if c not in new_ml_cols]

# 3. Reorder: Put the ML target/confidence first, then the original features
df_final = df[new_ml_cols + remaining_cols]

# 4. Save to CSV
df_final.to_csv('phishing_processed_v1.csv', index=False)

print("Success! File saved as 'phishing_processed_v1.csv'")
print(f"Columns at the start of file: {list(df_final.columns)[:5]}")

Success! File saved as 'phishing_processed_v1.csv'
Columns at the start of file: ['final_label', 'label_confidence', 'label_type', 'num_words', 'num_unique_words']


In [19]:
# 1. Check Final Class Balance
print("--- Final Class Distribution (Target Variable) ---")
print(df_final['final_label'].value_counts(normalize=True))

# 2. Check Feature Correlations (The "Signal")
# We only care about numeric columns correlating with the label
numeric_cols = ['num_words', 'num_unique_words', 'num_stopwords', 
                'num_links', 'num_unique_domains', 'num_email_addresses', 
                'num_spelling_errors', 'num_urgent_keywords']

# Ensure they are numeric
for c in numeric_cols:
    df_final[c] = pd.to_numeric(df_final[c], errors='coerce').fillna(0)

# Calculate correlation with the target
correlations = df_final[numeric_cols].corrwith(df_final['final_label']).sort_values(ascending=False)

print("\n--- Top Predictors (Correlation with Final Label) ---")
print(correlations)

--- Final Class Distribution (Target Variable) ---
final_label
0.0    0.987379
1.0    0.012621
Name: proportion, dtype: float64

--- Top Predictors (Correlation with Final Label) ---
num_urgent_keywords    0.029604
num_unique_words       0.020098
num_unique_domains     0.002620
num_stopwords          0.002221
num_words              0.001825
num_spelling_errors    0.000414
num_links             -0.000842
num_email_addresses   -0.011757
dtype: float64


In [20]:
# 1. Define the categories to check
expected_types = [
    'consensus_phish', 
    'consensus_safe', 
    'adjudicated_by_label3', 
    'single_judge', 
    'tie_needs_adjudication', 
    'no_judges'
]

# 2. Use df_final (The exact object being saved)
counts = df_final['label_type'].value_counts()
total_rows = len(df_final)

print(f"{'LABEL TYPE':<25} | {'COUNT':<10} | {'STATUS'}")
print("-" * 55)

for label in expected_types:
    count = counts.get(label, 0)
    
    # Determine Status
    if label == 'tie_needs_adjudication' and count == 0:
        status = "PASS (No Conflicts)"
    elif label == 'no_judges' and count == 0:
        status = "PASS (100% Coverage)"
    elif count > 0:
        status = "OK"
    else:
        status = "Check Logic"
        
    print(f"{label:<25} | {count:>10,} | {status}")

print("-" * 55)
print(f"{'TOTAL':<25} | {total_rows:>10,}")
print(f"\nColumns in Final File: {list(df_final.columns)[:5]} ...")

LABEL TYPE                | COUNT      | STATUS
-------------------------------------------------------
consensus_phish           |      5,608 | OK
consensus_safe            |     18,876 | OK
adjudicated_by_label3     |      2,465 | OK
single_judge              |    497,897 | OK
tie_needs_adjudication    |          0 | PASS (No Conflicts)
no_judges                 |          0 | PASS (100% Coverage)
-------------------------------------------------------
TOTAL                     |    524,846

Columns in Final File: ['final_label', 'label_confidence', 'label_type', 'num_words', 'num_unique_words'] ...


In [21]:
# 2. Count the values in 'final_label'
# 0 = Safe
# 1 = Phishing
counts = df['final_label'].value_counts()

print("--- Final Label Counts ---")
print(f"Safe (0):     {counts.get(0, 0):,}")
print(f"Phishing (1): {counts.get(1, 0):,}")

# Optional: Get the percentage breakdown
print("\n--- Percentages ---")
print(df['final_label'].value_counts(normalize=True).mul(100).round(2))

--- Final Label Counts ---
Safe (0):     518,222
Phishing (1): 6,624

--- Percentages ---
final_label
0.0    98.74
1.0     1.26
Name: proportion, dtype: float64


In [22]:
# Get the counts of each confidence level
# sort_index() keeps them in order (0.5, 0.66, 1.0)
conf_counts = df['label_confidence'].value_counts().sort_index()

print("--- Label Confidence Distribution ---")
print(conf_counts)

print("\n--- Percentages ---")
# Show as neat percentages
print(df['label_confidence'].value_counts(normalize=True).mul(100).sort_index().round(2))

--- Label Confidence Distribution ---
label_confidence
0.500000    497897
0.666667      1016
0.666667      1449
1.000000     24484
Name: count, dtype: int64

--- Percentages ---
label_confidence
0.500000    94.87
0.666667     0.19
0.666667     0.28
1.000000     4.66
Name: proportion, dtype: float64
